In [0]:
# Read checkpoint
#       ↓
# Read Silver CDF
#       ↓
# Keep insert + update_postimage
#       ↓
# Deduplicate if necessary
#       ↓
# Transform
#       ↓
# Validate dimension keys
#       ↓
# MERGE fact_orders
#       ↓
# Update daily_sales
#       ↓
# Advance checkpoint


# CDF history is transient and tied to Delta retention. If a job falls behind until its required starting version has been removed, that old CDF range is no longer readable.

In [0]:
%sql
DESCRIBE TABLE sentinel_dev.gold.fact_orders;

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("catalog", "sentinel_dev")

CATALOG = dbutils.widgets.get("catalog")

SILVER_TABLE = f"{CATALOG}.silver.silver_orders_current"
GOLD_TABLE = f"{CATALOG}.gold.fact_orders"

CUSTOMER_DIM = f"{CATALOG}.gold.dim_customer"
PRODUCT_DIM = f"{CATALOG}.gold.dim_product"

CHECKPOINT_TABLE = f"{CATALOG}.monitoring.gold_cdf_checkpoint"

PIPELINE_NAME = "fact_orders"

print(f"Environment catalog: {CATALOG}")

In [0]:
checkpoint_row = (
    spark.table(CHECKPOINT_TABLE)
        .filter(F.col("pipeline_name") == PIPELINE_NAME)
        .select("last_processed_version")
        .first()
)

if checkpoint_row is None:
    raise RuntimeError(
        f"Missing checkpoint row for {PIPELINE_NAME}"
    )

last_processed_version = checkpoint_row["last_processed_version"]

if last_processed_version is None:
    raise RuntimeError(
        "CDF checkpoint has not been bootstrapped."
    )

current_silver_version = (
    spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}")
        .agg(F.max("version").alias("version"))
        .first()["version"]
)

starting_version = last_processed_version + 1
ending_version = current_silver_version

print(f"Last processed : {last_processed_version}")
print(f"Starting       : {starting_version}")
print(f"Ending         : {ending_version}")

In [0]:
has_changes = starting_version <= ending_version

if not has_changes:
    print("No new Silver commits. Nothing to process.")

In [0]:
if has_changes:
    cdf_df = (
        spark.read
            .format("delta")
            .option("readChangeFeed", "true")
            .option("startingVersion", starting_version)
            .option("endingVersion", ending_version)
            .table(SILVER_TABLE)
    )

    relevant_changes_df = (
        cdf_df
            .filter(
                F.col("_change_type").isin(
                    "insert",
                    "update_postimage"
                )
            )
    )

    print(
        f"CDF rows to process: "
        f"{relevant_changes_df.count():,}"
    )

In [0]:
from pyspark.sql.window import Window

if has_changes:

    latest_change_window = (
        Window
            .partitionBy("order_id")
            .orderBy(
                F.col("_commit_version").desc(),
                F.col("order_timestamp_clean").desc()
            )
    )

    latest_changes_df = (
        relevant_changes_df
            .withColumn(
                "_rn",
                F.row_number().over(latest_change_window)
            )
            .filter(F.col("_rn") == 1)
            .drop("_rn")
    )

In [0]:
if has_changes:

    customer_df = spark.table(CUSTOMER_DIM)
    product_df = spark.table(PRODUCT_DIM)

    gold_changes_df = (
        latest_changes_df.alias("o")

        .join(
            customer_df.alias("c"),
            F.col("o.customer_id") == F.col("c.customer_id"),
            "left"
        )

        .join(
            product_df.alias("p"),
            F.col("o.product_id") == F.col("p.product_id"),
            "left"
        )

        .select(
            F.col("o.order_id"),
            F.col("c.customer_key"),
            F.col("p.product_key"),

            F.to_date(
                F.col("o.order_timestamp_clean")
            ).alias("order_date"),

            F.col("o.order_timestamp_clean")
                .alias("order_timestamp"),

            F.col("o.quantity_clean")
                .alias("quantity"),

            F.col("o.unit_price_clean")
                .alias("unit_price"),

            F.col("o.total_amount_clean")
                .alias("total_amount"),
            F.col("o.payment_method"),
            F.col("o.order_status"),
            F.col("o.source_system"),
            F.col("o.ingested_at")
        )
    )

In [0]:
if has_changes:

    unresolved_dimensions = (
        gold_changes_df
            .filter(
                F.col("customer_key").isNull()
                | F.col("product_key").isNull()
            )
            .count()
    )

    if unresolved_dimensions > 0:
        raise RuntimeError(
            f"{unresolved_dimensions} Gold rows have "
            "unresolved dimension keys."
        )

    print("Dimension validation passed.")

In [0]:
if has_changes:

    gold_delta = DeltaTable.forName(
        spark,
        GOLD_TABLE
    )

    (
        gold_delta.alias("target")
            .merge(
                gold_changes_df.alias("source"),
                "target.order_id = source.order_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
    )

    print("Gold MERGE successful.")

In [0]:
if has_changes:

    spark.sql(f"""
        UPDATE {CHECKPOINT_TABLE}

        SET last_processed_version = {ending_version}

        WHERE pipeline_name = '{PIPELINE_NAME}'
    """)

    print(
        f"Checkpoint advanced to Silver version "
        f"{ending_version}."
    )

In [0]:
print("=" * 50)
print("SENTINEL INCREMENTAL GOLD")
print("=" * 50)

if has_changes:
    print(f"Processed versions : {starting_version}-{ending_version}")
    print("Status             : SUCCESS")
else:
    print("New Silver commits : 0")
    print("Status             : NO CHANGES")

print("=" * 50)